# Tutorial 02 — Gasificador 0D: Pirólisis con alivio de presión (Semibatch)

**Equipo:** Gasificador de lecho fijo · **Modo:** 0D (N=1) · Semibatch (venteo controlado por presión)  
**Fenómeno:** Pirólisis de biomasa con acumulación de gas y alivio por válvula reguladora de presión  
**Prerrequisito:** Tutorial 01 — Gasificador 0D Batch

## Introducción

Este tutorial extiende el caso batch del Tutorial 01 añadiendo una **válvula de alivio de presión**
en la salida del reactor. El resto del sistema es idéntico: reactor 0D, biomasa seca, calefacción
externa a 800 °C, sin entrada de agente gasificante.

### Diferencia respecto al modo Batch

| Parámetro | Batch (Tutorial 01) | Semibatch (este tutorial) |
|-----------|--------------------|-----------------------------|
| `v_gas_in` | `None` | `None` |
| `v_out`    | `0.0` (sellado) | `> 0` (venteo activo) |
| Comportamiento de la salida | Sistema sellado: P sube libremente | Válvula cerrada hasta que P > P_out; abre proporcionalmente al exceso de presión |

### Ley de la válvula de alivio

En semibatch (`v_out > 0`), la velocidad de salida del gas se calcula como:

$$v_{out}(t) = \max\\!\\left(0,\\; \\frac{P(t) - P_{out}}{P_{out}}\\right) \\cdot v_{out}$$

donde `v_out` [m/s] es el parámetro pasado a `build_bc_config` y actúa como la velocidad
máxima de venteo (apertura total de válvula cuando P = 2·P_out).

| Condición | Comportamiento |
|-----------|----------------|
| $P = P_{out}$ | Válvula cerrada ($v_{out,actual} = 0$) |
| $P = 2 \cdot P_{out}$ | Venteo máximo ($v_{out,actual} = v_{out}$) |
| `v_out` pequeño | Válvula restrictiva → P sube mucho antes de ventilar |
| `v_out` grande | Válvula permeable → P ≈ P_out en todo momento (≈ batch sellado a baja P) |

### Dos casos comparados

| Caso | `v_out` [m/s] | Descripción física |
|------|---------------|-----------------------|
| **A — Venteo lento** | 0.01 | Válvula restrictiva. El gas producido se acumula; la presión puede subir varios bar. |
| **B — Venteo rápido** | 0.50 | Válvula permeable. P se mantiene próxima a la atmosférica. |

### Qué observar

1. **Presión** ($P$ vs $t$): cuánto sube en el caso lento; cómo permanece casi plana en el rápido.
2. **Velocidad de venteo** ($v_{out,actual}$ vs $t$): el perfil temporal de apertura de la válvula.
3. **Temperaturas** ($T_g$, $T_s$ vs $t$): el gas acumulado tiene mayor masa → diferente dinámica térmica.
4. **Composición del gas**: las fracciones molares son similares (mismas reacciones), pero las concentraciones absolutas [mol/m³] serán mucho mayores en el caso lento.

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt

# Ajustar ruta al raíz del proyecto (test/gasifier/ → ../../)
ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from src.io.fuels_reader import read_fueldb
from src.units.gasifier.config.gas_props    import build_gas_prop_config, GASIFIER_GAS_SPECIES
from src.units.gasifier.config.solid_props  import build_solid_prop_config
from src.units.gasifier.config.transport    import build_transport_config
from src.units.gasifier.config.thermal_bc   import build_thermal_bc_config
from src.units.gasifier.config.boundary_c   import build_bc_config
from src.units.gasifier.config.initial_c    import build_initial_c_config
from src.solvers.runner_gasifier            import run_step
from src.postprocessing.gasifier_balances   import check_balances, display_balances
from src.postprocessing.gasifier_plots      import (
    plot_temperatures, plot_solid_evolution,
    plot_gas_composition, plot_pressure,
)

print(f"ROOT: {ROOT}")

## 1. Bases de datos y propiedades del combustible

In [ ]:
# Rutas a bases de datos
FUEL_PATH = os.path.join(ROOT, "materials", "fuels", "softwood_spruce.yaml")
GAS_DB    = os.path.join(ROOT, "materials", "fluids", "gasdb.txt")
SOLID_DB  = os.path.join(ROOT, "materials", "solids", "soliddb.txt")

# Carga del combustible
fuel_config = read_fueldb(FUEL_PATH)

# Resumen del combustible cargado
rho_p = float(fuel_config["physical"]["rho_particle"])   # [kg/m³]
dp0   = float(fuel_config["physical"]["dp_initial"])     # [m]
print(f"Combustible : {fuel_config['description']}")
print(f"ρ_partícula : {rho_p:.0f} kg/m³")
print(f"dp_inicial  : {dp0*1000:.1f} mm")

## 2. Geometría y condiciones de operación

Mismo reactor del Tutorial 01: tubo de ≈ 785 cm³ calentado externamente.
Geometría y carga de sólido idénticas para que la comparación sea directa.

In [ ]:
# ── Geometría ─────────────────────────────────────────────────────────────────
N  = 1       # Número de celdas → modelo 0D (volumen perfectamente mezclado)
L  = 0.10    # [m]  longitud del reactor
Di = 0.10    # [m]  diámetro interno
Do = 0.114   # [m]  diámetro externo (e_pared = 7 mm, acero inox 316L)

dz     = L / N                     # [m]   tamaño de celda
Ai     = 0.25 * np.pi * Di**2      # [m²]  sección transversal interna
Pi     = np.pi * Di                # [m]   perímetro interno
Po     = np.pi * Do                # [m]   perímetro externo
e_wall = (Do - Di) / 2             # [m]   espesor de pared

# ── Condiciones de operación ──────────────────────────────────────────────────
epsi_r = 0.60       # [-]   porosidad del lecho (fracción de huecos gas/volumen lecho)
P_out  = 1.01325    # [bar] presión de referencia (salida de la válvula de alivio)

# ── Carga inicial de biomasa ──────────────────────────────────────────────────
mc_wb       = 0.15                                          # [-]         contenido de humedad base húmeda
rho_bio_dry = rho_p * (1 - epsi_r)                          # [kg/m³_bed] biomasa seca en lecho
rho_moi_0   = rho_bio_dry * mc_wb / (1 - mc_wb)             # [kg/m³_bed] humedad inicial
rho_bio_0   = rho_bio_dry                                   # [kg/m³_bed] biomasa inicial
rho_char_0  = 1e-6                                          # [kg/m³_bed] traza de char inicial

# Densidad de referencia del char (para modelo de partícula decreciente SCM)
rho_char0 = fuel_config["pyrolysis_yields"]["char"] * rho_p * (1 - epsi_r)

print(f"Geometría:")
print(f"  N = {N} celdas → modelo 0D")
print(f"  V_reactor = {Ai*L*1e6:.1f} cm³  (Di={Di*100:.0f} cm, L={L*100:.0f} cm)")
print(f"\nCarga inicial de sólido:")
print(f"  ρ_biomasa  = {rho_bio_0:.1f} kg/m³_bed")
print(f"  ρ_humedad  = {rho_moi_0:.1f} kg/m³_bed  ({mc_wb*100:.0f} % wb)")
print(f"  ρ_char_ini = {rho_char_0:.1e} kg/m³_bed  (traza)")

## 3. Condiciones iniciales del gas

Igual que en el Tutorial 01: N₂ puro a temperatura ambiente (300 K) y presión atmosférica.

In [ ]:
T_init = 300.0    # [K] temperatura inicial de gas y sólido (≈ 27 °C)

species = list(GASIFIER_GAS_SPECIES)   # ['CO','CO2','H2O','H2','O2','CH4','C2H4','tar','N2']
nc      = len(species)                 # 9 especies

y0 = np.zeros(nc)
y0[species.index("N2")] = 1.0    # 100 % N₂ al inicio

print(f"Temperatura inicial: {T_init:.1f} K ({T_init-273.15:.1f} °C)")
print(f"Composición inicial: 100 % N₂")
print(f"Presión inicial    : {P_out:.5f} bar")

## 4. Condición térmica (común a ambos casos)

Pared a temperatura fija de 800 °C, igual que en el Tutorial 01.
La condición térmica no varía entre los dos casos — solo cambia el modo de la salida del gas.

In [ ]:
T_wall   = 1073.15    # [K] = 800 °C — temperatura de la pared exterior
k_wall   = 16.5       # [W/m/K]  conductividad térmica acero inoxidable 316L
rho_wall = 7950.0     # [kg/m³]  densidad acero inoxidable 316L
Cp_wall  = 510.0      # [J/kg/K] calor específico acero inoxidable 316L

tbc_cfg = build_thermal_bc_config(
    mode     = "fixed_twall",
    Di       = Di,
    Do       = Do,
    e_wall   = e_wall,
    T_wall   = T_wall,
    k_wall   = k_wall,
    rho_wall = rho_wall,
    Cp_wall  = Cp_wall,
)

print(f"BC térmica: pared fija  T_wall = {T_wall:.2f} K ({T_wall-273.15:.1f} °C)")
print(f"Material  : k={k_wall} W/m/K  ρ={rho_wall:.0f} kg/m³  Cp={Cp_wall:.0f} J/kg/K")

## 5. Propiedades y modelo de transporte

In [ ]:
# ── Propiedades del gas (modo polinomial: Cp, µ, k dependientes de T) ──────────
prop_gas  = build_gas_prop_config(fuel_config=fuel_config, mode="polynomial", db_path=GAS_DB)
gas_T_ref = float(np.min(np.asarray(prop_gas["Tref"])))   # [K] temperatura de referencia de entalpía
MW_arr    = np.asarray(prop_gas["MW"])                     # [kg/mol]

# ── Propiedades del sólido (Cp_fns y h_fns como funciones de T) ────────────────
solid_config = build_solid_prop_config(fuel_config)

# ── Modelo de transporte: constante ────────────────────────────────────────────
trans_cfg = build_transport_config(
    mode   = "constant",
    N      = N,
    n_comp = nc,
    h_bed  = 80.0,    # [W/m²/K] coeficiente gas-partícula
    h_wall = 20.0,    # [W/m²/K] coeficiente gas-pared
)

print(f"gas_T_ref = {gas_T_ref:.2f} K  (referencia de entalpía)")
print(f"Transporte: constante  (h_bed=80, h_wall=20 W/m²/K)")

## 6. Vector de estado inicial (sv0)

El mismo sv0 sirve para los dos casos, ya que las condiciones iniciales son idénticas.

```
sv0 = [C(nc×N), ρ_s(3×N), Hg(N), Ts(N), Q_mt_acc(N), Q_rxn_acc(N), Q_gs_acc(N)]
      └── 9N ──┘└── 3N ──┘└─ N ┘└─ N ┘└────── N ─────┘└────── N ──┘└────── N ──┘
                                                                    total = 17×N
```

Para N=1: `sv0.shape = (17,)`

In [ ]:
init_cfg = build_initial_c_config(
    P_init            = P_out,
    Tg_init           = T_init,
    Ts_init           = T_init,
    y_init            = y0,
    rho_biomass_init  = rho_bio_0,
    rho_char_init     = rho_char_0,
    rho_moisture_init = rho_moi_0,
    n_comp            = nc,
    N                 = N,
    prop_gas          = prop_gas,
    epsi_r            = epsi_r,
    gas_T_ref         = gas_T_ref,
    Tw_init           = None,    # Sin modelo dinámico de pared
)

sv0 = init_cfg["sv0"]
print(f"sv0.shape = {sv0.shape}   ({sv0.shape[0] // N} × {N} = {sv0.shape[0]} DOFs)")

## 7. Parámetros base (comunes a ambos casos)

El diccionario `params_base` contiene todos los parámetros compartidos. La clave `bc_config`
se añadirá por separado en cada caso, ya que es el único parámetro que cambia.

In [ ]:
params_base = {
    # Geometría
    "n_comp" : nc,
    "N"      : N,
    "dz"     : dz,
    "Ai"     : Ai,
    "Di"     : Di,
    "Pi"     : Pi,
    "Po"     : Po,
    # Propiedades del gas
    "prop_gas"   : prop_gas,
    "MW"         : MW_arr,
    "gas_T_ref"  : gas_T_ref,
    # Configuraciones compartidas
    "trans_config"     : trans_cfg,
    "thermal_bc_config": tbc_cfg,
    "energy"           : True,
    # Específico del gasificador
    "epsi_r"      : epsi_r,
    "dp0"         : dp0,
    "rho_char0"   : rho_char0,
    "fuel_config" : fuel_config,
    "solid_config": solid_config,
    "species"     : species,
    # bc_config no está aquí → se añade en cada caso
}

---
## 8.A — Caso A: Venteo lento (`v_out = 0.01 m/s`)

La válvula de alivio tiene una capacidad muy limitada. El gas producido por la pirólisis
se acumula en el reactor y la presión puede subir varios bar antes de que la tasa de venteo
sea suficiente para compensar la producción.

**Nota sobre la fórmula:** con `v_out = 0.01` m/s, la válvula solo alcanza su apertura
máxima cuando $P = 2 \cdot P_{out} \approx 2$ bar. Para $P = 1.1 \cdot P_{out}$,
$v_{out,actual} = 0.001$ m/s — una décima parte del máximo.

In [ ]:
v_vent_lento = 0.01    # [m/s] velocidad máxima de venteo — válvula restrictiva

bc_cfg_A = build_bc_config(
    n_comp    = nc,
    P_out_bar = P_out,
    v_gas_in  = None,
    v_out     = v_vent_lento,   # venteo: v_out actúa como velocidad máxima de alivio
)

T_MAX = 600.0    # [s]
params_A = {**params_base, "bc_config": bc_cfg_A}

print("=" * 60)
print(f"Caso A — Venteo lento   v_out = {v_vent_lento} m/s")
print("=" * 60)

t_A, y_hist_A, gasifier_A = run_step(
    sv0           = sv0,
    t_max         = T_MAX,
    params        = params_A,
    rtol          = 1e-5,
    atol          = 1e-7,
    n_sec         = 5,
    show_progress = True,
)

if t_A[-1] < T_MAX * 0.99:
    print(f"\n⚠  El solver no alcanzó t_max. Último t = {t_A[-1]:.1f} s")
else:
    print(f"\n✓  Completado. t_final = {t_A[-1]:.1f} s  |  pasos = {len(t_A)}")
    P_max_A = float(np.max(gasifier_A._P_results[:, 0]))
    print(f"   P_max    = {P_max_A:.4f} bar  ({P_max_A/P_out:.2f} × P_out)")
    print(f"   Tg_final = {gasifier_A._Tg_results[-1, 0]:.1f} K  "
          f"({gasifier_A._Tg_results[-1, 0]-273.15:.1f} °C)")
    print(f"   Ts_final = {gasifier_A._Ts_results[-1, 0]:.1f} K  "
          f"({gasifier_A._Ts_results[-1, 0]-273.15:.1f} °C)")

### Resultados — Caso A

In [ ]:
fig, ax = plot_temperatures(
    gasifier_A,
    T_ref       = T_wall,
    T_ref_label = f"$T_{{pared}}$ = {T_wall - 273.15:.0f} °C",
)
ax.set_title(f"Temperaturas — Caso A (venteo lento, $v_{{vent}}$ = {v_vent_lento} m/s)")
plt.show()

In [ ]:
fig, ax = plot_solid_evolution(gasifier_A)
ax.set_title(f"Evolución de la fase sólida — Caso A")
plt.show()

In [ ]:
fig, ax = plot_gas_composition(gasifier_A)
ax.set_title(f"Composición del gas producido — Caso A")
plt.show()

In [ ]:
fig, ax = plot_pressure(gasifier_A)
ax.axhline(P_out, ls="--", color="gray", alpha=0.7, label=f"$P_{{out}}$ = {P_out:.3f} bar")
ax.legend(fontsize=9)
ax.set_title(f"Presión — Caso A (venteo lento, $v_{{vent}}$ = {v_vent_lento} m/s)")
plt.show()

---
## 8.B — Caso B: Venteo rápido (`v_out = 0.50 m/s`)

La válvula tiene una capacidad muy elevada. Cualquier pequeño exceso de presión sobre $P_{out}$
genera un caudal de venteo suficiente para evacuar el gas producido inmediatamente.
Se espera que la presión permanezca muy próxima a la atmosférica en todo momento.

**Único cambio respecto al Caso A:** `v_out = 0.50 m/s` en lugar de `0.01 m/s`.

In [ ]:
v_vent_rapido = 0.50    # [m/s] velocidad máxima de venteo — válvula permeable

bc_cfg_B = build_bc_config(
    n_comp    = nc,
    P_out_bar = P_out,
    v_gas_in  = None,
    v_out     = v_vent_rapido,  # venteo: v_out actúa como velocidad máxima de alivio
)

params_B = {**params_base, "bc_config": bc_cfg_B}

print("=" * 60)
print(f"Caso B — Venteo rápido  v_out = {v_vent_rapido} m/s")
print("=" * 60)

t_B, y_hist_B, gasifier_B = run_step(
    sv0           = sv0,
    t_max         = T_MAX,
    params        = params_B,
    rtol          = 1e-5,
    atol          = 1e-7,
    n_sec         = 5,
    show_progress = True,
)

if t_B[-1] < T_MAX * 0.99:
    print(f"\n⚠  El solver no alcanzó t_max. Último t = {t_B[-1]:.1f} s")
else:
    print(f"\n✓  Completado. t_final = {t_B[-1]:.1f} s  |  pasos = {len(t_B)}")
    P_max_B = float(np.max(gasifier_B._P_results[:, 0]))
    print(f"   P_max    = {P_max_B:.4f} bar  ({P_max_B/P_out:.4f} × P_out)")
    print(f"   Tg_final = {gasifier_B._Tg_results[-1, 0]:.1f} K  "
          f"({gasifier_B._Tg_results[-1, 0]-273.15:.1f} °C)")
    print(f"   Ts_final = {gasifier_B._Ts_results[-1, 0]:.1f} K  "
          f"({gasifier_B._Ts_results[-1, 0]-273.15:.1f} °C)")

### Resultados — Caso B

In [ ]:
fig, ax = plot_temperatures(
    gasifier_B,
    T_ref       = T_wall,
    T_ref_label = f"$T_{{pared}}$ = {T_wall - 273.15:.0f} °C",
)
ax.set_title(f"Temperaturas — Caso B (venteo rápido, $v_{{vent}}$ = {v_vent_rapido} m/s)")
plt.show()

In [ ]:
fig, ax = plot_pressure(gasifier_B)
ax.axhline(P_out, ls="--", color="gray", alpha=0.7, label=f"$P_{{out}}$ = {P_out:.3f} bar")
ax.legend(fontsize=9)
ax.set_title(f"Presión — Caso B (venteo rápido, $v_{{vent}}$ = {v_vent_rapido} m/s)")
plt.show()

---
## 9. Comparación venteo lento vs venteo rápido

Las gráficas siguientes superponen los dos casos para evidenciar el efecto de la capacidad
de venteo en cada variable del proceso.

In [ ]:
# ── Presión y velocidad de venteo ────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

# — Presión —
axes[0].plot(t_A, gasifier_A._P_results[:, 0],
             lw=2, color="steelblue",  label=f"Lento  ($v_{{vent}}$ = {v_vent_lento} m/s)")
axes[0].plot(t_B, gasifier_B._P_results[:, 0],
             lw=2, color="darkorange", label=f"Rápido ($v_{{vent}}$ = {v_vent_rapido} m/s)",
             ls="--")
axes[0].axhline(P_out, ls=":", color="gray", alpha=0.7, label=f"$P_{{out}}$ = {P_out:.3f} bar")
axes[0].set_ylabel("Presión [bar]")
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)
axes[0].set_title("Presión en el reactor")

# — Velocidad de venteo (v_out) —
axes[1].plot(t_A, gasifier_A._v_out_results,
             lw=2, color="steelblue",  label=f"Lento  ($v_{{vent}}$ = {v_vent_lento} m/s)")
axes[1].plot(t_B, gasifier_B._v_out_results,
             lw=2, color="darkorange", label=f"Rápido ($v_{{vent}}$ = {v_vent_rapido} m/s)",
             ls="--")
axes[1].set_xlabel("Tiempo [s]")
axes[1].set_ylabel("$v_{out}$ [m/s]")
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)
axes[1].set_title("Velocidad de venteo (salida de la válvula de alivio)")

fig.tight_layout()
plt.show()

In [ ]:
# ── Temperaturas de gas y sólido ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

for ax, col, label, color in [
    (axes[0], gasifier_A, f"Venteo lento ({v_vent_lento} m/s)",  "steelblue"),
    (axes[1], gasifier_B, f"Venteo rápido ({v_vent_rapido} m/s)", "darkorange"),
]:
    t_    = col._t_results
    Tg_C  = col._Tg_results[:, 0] - 273.15
    Ts_C  = col._Ts_results[:, 0] - 273.15
    ax.plot(t_, Tg_C, lw=2, color=color,         label="$T_g$ (gas)")
    ax.plot(t_, Ts_C, lw=2, color=color, ls="--", label="$T_s$ (sólido)")
    ax.axhline(T_wall - 273.15, ls=":", color="red", alpha=0.6, label="$T_{pared}$")
    ax.set_xlabel("Tiempo [s]")
    ax.set_ylabel("Temperatura [°C]")
    ax.set_title(label)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

fig.suptitle("Comparación de temperaturas", fontweight="bold")
fig.tight_layout()
plt.show()

In [ ]:
# ── Composición del gas — fracciones molares ──────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

_, _ = plot_gas_composition(gasifier_A, ax=axes[0])
axes[0].set_title(f"Composición gas — venteo lento ({v_vent_lento} m/s)")

_, _ = plot_gas_composition(gasifier_B, ax=axes[1])
axes[1].set_title(f"Composición gas — venteo rápido ({v_vent_rapido} m/s)")

fig.suptitle("Fracciones molares del gas producido", fontweight="bold")
fig.tight_layout()
plt.show()

In [ ]:
# ── Concentración molar total del gas ─────────────────────────────────────────
# La diferencia entre casos es máxima aquí: más presión → más mol/m³
R_GAS = 8.31446261815324   # [J/mol/K]

# Concentración total = P [Pa] / (R · Tg [K])
Ctot_A = (gasifier_A._P_results[:, 0] * 1e5) / (R_GAS * gasifier_A._Tg_results[:, 0])
Ctot_B = (gasifier_B._P_results[:, 0] * 1e5) / (R_GAS * gasifier_B._Tg_results[:, 0])

fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(t_A, Ctot_A, lw=2, color="steelblue",  label=f"Lento  ({v_vent_lento} m/s)")
ax.plot(t_B, Ctot_B, lw=2, color="darkorange", label=f"Rápido ({v_vent_rapido} m/s)", ls="--")
ax.set_xlabel("Tiempo [s]")
ax.set_ylabel("$C_{tot}$ [mol/m³_gas]")
ax.set_title("Concentración molar total del gas en el reactor")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

print(f"Ctot máxima — Caso A (lento):  {float(np.max(Ctot_A)):.1f} mol/m³")
print(f"Ctot máxima — Caso B (rápido): {float(np.max(Ctot_B)):.1f} mol/m³")

---
## 10. Verificación de balances

Los balances deben cerrarse en los dos casos. El modo de venteo no rompe los balances:
en el Caso A hay más flujo de salida tardío; en el B la salida ocurre desde el principio.

In [ ]:
print(f"{'='*60}")
print(f"Balances — Caso A (venteo lento, v_out = {v_vent_lento} m/s)")
print(f"{'='*60}")
balances_A = check_balances(gasifier_A, params_A, verbose=False)
display_balances(balances_A)

In [ ]:
print(f"{'='*60}")
print(f"Balances — Caso B (venteo rápido, v_out = {v_vent_rapido} m/s)")
print(f"{'='*60}")
balances_B = check_balances(gasifier_B, params_B, verbose=False)
display_balances(balances_B)

## Interpretación de balances

**★ Cierre numérico** — residual debe ser ≈ 0 (umbral: < 1 % del término mayor)  
**~ Residual físico** — residual = cantidad producida o consumida; se espera ≠ 0, es información útil

### Convención de símbolos

| Símbolo | Tipo | Debe ser |
|---------|------|----------|
| ★ | Cierre numérico (masa total, energía gas, energía sólido, balance global) | ≈ 0 |
| ~ | Residual físico (especies gas, masa sólida) | ≠ 0 — muestra cuánto reaccionó |
| ✓ OK | Cierre ★ dentro del umbral (< 1 %) | — |
| ⚠ REVISAR | Cierre ★ fuera del umbral | Investigar causa raíz |

### Diferencias respecto al Tutorial 01 (batch sellado)

| Balance | Batch sellado | Semibatch lento | Semibatch rápido |
|---------|---------------|-----------------|------------------|
| ★ Masa total | Flujo_masa = 0 (nada sale) | Flujo_masa ≠ 0 (gas ventea) | Flujo_masa ≠ 0 |
| ★ Energía gas | Fh_conv_neto = 0 | Fh_conv_neto ≠ 0 (entalpía del gas ventado) | Fh_conv_neto ≠ 0 |
| ★ Energía sólido | ≈ 0 | ≈ 0 | ≈ 0 |
| ★ Balance global | ≈ 0 | ≈ 0 | ≈ 0 |

En el modo `vent`, `Fh_conv_neto ≠ 0` porque el gas que ventea lleva consigo entalpía.
El balance global sigue cerrando porque esa entalpía sale del lado derecho de la ecuación.

### Si algún balance ★ no cierra (> 1 %)

- **Masa total > 1 %**: verificar que el flujo de venteo `v_out × Ctot_out × epsi_r`
  se integra correctamente en el balance másico.
- **Energía gas > 1 %**: verificar que el acumulador `Q_mt_acc` está activo
  y que la entalpía del gas ventado está incluida en `Fh_conv_neto`.
- **Energía sólido > 1 %**: verificar `solid_config["h_fns"]` (integrales ∫Cp dT).
- **Balance global > 1 %**: normalmente implica que uno de los dos anteriores falla.

## Conclusiones

### Qué demuestra este tutorial

1. **Modo semibatch**: usar `v_out > 0` en `build_bc_config` introduce una dinámica
   de presión que no existe en el batch sellado (`v_out = 0.0`).

2. **La válvula controla la presión, no la reacción**: las reacciones (pirólisis, secado)
   ocurren a la misma temperatura y siguen la misma cinética en ambos casos.
   Lo que cambia es cuánto gas se acumula y a qué presión opera el reactor.

3. **Venteo lento → acumulación de gas**: la concentración molar del gas sube muy
   por encima de la concentración atmosférica. El gas producido tiene más masa y más
   capacidad calorífica efectiva.

4. **Venteo rápido → comportamiento próximo al batch sellado a baja presión**: con `v_out`
   suficientemente grande, cualquier pequeño exceso de presión genera un caudal de venteo
   que mantiene $P \approx P_{out}$.

5. **Fracciones molares similares, concentraciones distintas**: dado que las reacciones
   son las mismas, la composición del gas (fracciones) es similar en ambos casos.
   Sin embargo, las concentraciones absolutas [mol/m³] son mucho mayores en el caso lento.

6. **Los balances cierran en ambos casos**: el modo semibatch no rompe los cierres.
   En semibatch, `Fh_conv_neto ≠ 0` porque el gas que sale lleva entalpía consigo.

### Qué probar a continuación

- **Cambiar `T_wall`**: ¿a qué temperatura mínima arranca la pirólisis?
  ¿Cómo afecta la velocidad de conversión del sólido? Fijar `v_out` en un valor
  intermedio (~0.05 m/s) para mantener P razonable mientras se varía T_wall.

- **Cambiar `mc_wb`**: efecto del contenido de humedad en el balance energético.
  Con `mc_wb` alto el secado domina la primera fase; con `mc_wb ≈ 0` la pirólisis
  arranca antes. ¿Cómo cambia la temperatura de sólido y la composición del gas?

- **Cambiar `T_MAX`**: ¿cuándo se agota la biomasa? ¿Qué ocurre cuando rho_biomasa → 0?
  Simular con `T_MAX = 1800 s` y observar si el sólido desaparece antes del final.

- **Barrer `v_out`**: ¿a qué valor el semibatch se vuelve indistinguible del batch sellado?

- **Presión de consigna más alta** (`P_out = 2 bar`): ¿cómo cambia la pirólisis a mayor presión?

- **Pasar a CSTR** (Tutorial 03): ¿qué ocurre si además se inyecta un agente gasificante?

- **Pasar a 1D** (Tutorial 10): ¿qué gradientes axiales aparecen con N=10 en modo batch?